# rank-world-size-args — ex2: reduce-protocol — (tensor, rank, world_size, dst=0)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rank-world-size-args`. Running the final beacon cell reports progress against the `Distributed: rank/world_size args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank/world_size args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank-world-size-args`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank-world-size-args"
DD_SUBTOPIC = "Distributed: rank/world_size args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Reduce-protocol signature — `(tensor, rank, world_size, dst=0, op)`

Ex1 implemented the broadcast-protocol (one source, fan out). The reduce-protocol is the dual — many sources, fan IN to a single dst:

```python
def reduce_protocol(tensor, rank, world_size, dst=0, op='sum'):
    if rank != dst:
        return [('send', dst)]
    # dst rank receives from every other rank and aggregates
    return [('recv', other) for other in range(world_size) if other != dst]
```

**Why the signature differs.** `src` becomes `dst`. Reasoning: in broadcast the SOURCE rank has the canonical data; in reduce the DESTINATION rank ends up with the canonical data. The kwarg names the canonical-data rank's role in each topology.

**Why ascending `other_rank` order is the convention.** Same as ex1's broadcast — order matters for deterministic test assertions and matches the order `dist.reduce`'s ring traversal uses internally.

**The `op` arg is unused by the protocol logic.** It only matters for the actual aggregation (sum vs max vs ...). The protocol is topology-only — same `(send, recv)` shape for any reduction op.

### Exercise 2 — reduce-protocol — (tensor, rank, world_size, dst=0)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `(tensor, rank, world_size, dst=0)` signature convention for reduce — branch on `rank != dst` to decide between sending and receiving, listing `range(world_size)` for the fan-in.
> Keywords: rank, world_size, dst, reduce, protocol-signature
> ```

**KCs targeted:** `rank-world-size-positional-args`, `dst-rank-branching`

Implement `ex2_reduce_protocol(tensor, rank, world_size, dst=0)` — a pure-logic dual of ex1's broadcast-protocol. Returns the list of `(action, other_rank)` tuples describing what THIS rank would do in a `reduce` collective.

Spec:
- If `rank == dst`: for each `other_rank in range(world_size)` where `other_rank != dst`, append `('recv', other_rank)`. Order matters — ascending by `other_rank`.
- If `rank != dst`: return `[('send', dst)]`.
- `tensor` is unused — only there to mirror the real signature.

Return the list of `(action, other_rank)` tuples.

This is the dual of ex1's broadcast-protocol:
- ex1 (broadcast): rank == SRC sends; rank != src recvs from src.
- ex2 (reduce): rank != DST sends to dst; rank == dst recvs from all others.
Same plumbing, opposite arrow.

In [ ]:
def ex2_reduce_protocol(tensor: Tensor, rank: int, world_size: int, dst: int = 0) -> list:
    """Pure-logic reduce protocol. Returns list[(action, other_rank)]."""
    raise NotImplementedError()


def _test_ex2():
    dummy = t.tensor([1.0])

    # rank 0 = dst, world_size=3 → recvs from 1 and 2.
    assert ex2_reduce_protocol(dummy, rank=0, world_size=3, dst=0) == [
        ('recv', 1), ('recv', 2)
    ]
    # rank 1 (not dst) → sends to dst=0.
    assert ex2_reduce_protocol(dummy, rank=1, world_size=3, dst=0) == [('send', 0)]
    assert ex2_reduce_protocol(dummy, rank=2, world_size=3, dst=0) == [('send', 0)]

    # Custom dst=2, world_size=4 → rank 2 recvs from 0,1,3; others send to 2.
    assert ex2_reduce_protocol(dummy, rank=2, world_size=4, dst=2) == [
        ('recv', 0), ('recv', 1), ('recv', 3)
    ]
    for non_dst in [0, 1, 3]:
        assert ex2_reduce_protocol(dummy, rank=non_dst, world_size=4, dst=2) == [('send', 2)]

    # Single-rank degenerate world_size=1 → dst is the only rank, no recvs.
    assert ex2_reduce_protocol(dummy, rank=0, world_size=1, dst=0) == []

    # Signature check: dst defaults to 0, positional order matches convention.
    import inspect
    sig = inspect.signature(ex2_reduce_protocol)
    params = list(sig.parameters.values())
    names = [p.name for p in params]
    assert names == ['tensor', 'rank', 'world_size', 'dst'], f'signature order wrong: {names}'
    assert sig.parameters['dst'].default == 0, 'dst must default to 0'

    # Action labels are exactly 'send' and 'recv' — not 'reduce', 'forward', etc.
    for r in range(5):
        actions = ex2_reduce_protocol(dummy, rank=r, world_size=5, dst=0)
        for action, _ in actions:
            assert action in ('send', 'recv'), f'unexpected action label {action!r}'

    # Cross-check duality with ex1's broadcast-protocol if it's defined:
    # A broadcast(src=k) has rank k as the SENDER (rank == src).
    # A reduce(dst=k) has rank k as the RECEIVER (rank == dst).
    # So reduce(dst=k) on rank k yields all 'recv' actions; broadcast(src=k) on rank k yields all 'send'.
    world_size = 5
    for k in range(world_size):
        actions_at_dst = ex2_reduce_protocol(dummy, rank=k, world_size=world_size, dst=k)
        # All recvs, one per other rank.
        assert all(a == 'recv' for a, _ in actions_at_dst)
        assert len(actions_at_dst) == world_size - 1
        # Sorted ascending by other_rank.
        sorted_others = sorted(o for _, o in actions_at_dst)
        assert [o for _, o in actions_at_dst] == sorted_others

    # Edge case: world_size=2, dst=1.
    assert ex2_reduce_protocol(dummy, rank=0, world_size=2, dst=1) == [('send', 1)]
    assert ex2_reduce_protocol(dummy, rank=1, world_size=2, dst=1) == [('recv', 0)]
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_reduce_protocol(tensor: Tensor, rank: int, world_size: int, dst: int = 0) -> list:
    if rank == dst:
        return [('recv', other) for other in range(world_size) if other != dst]
    return [('send', dst)]
```

**Broadcast vs reduce — same structure, opposite arrow.** Both have the `(tensor, rank, world_size, <canonical_rank>=0)` signature. In broadcast, the canonical rank is `src` (it OWNS the data others copy). In reduce, the canonical rank is `dst` (it WILL OWN the aggregated data). The branch flips: `rank == src` sends in broadcast; `rank == dst` recvs in reduce.

**`all_reduce` drops both kwargs.** Every rank is both source and destination, so `all_reduce(tensor, rank, world_size, op)` — no `src` or `dst` needed. Same for `all_gather`. The presence of a single-rank arg is the signature signal that the topology has a canonical-data rank.

**Why list-of-tuples is the right return type.** A `(rank, world_size, dst)` triple maps to a SET of point-to-point actions. Returning the list lets the test enumerate them in deterministic order without parsing prose.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()